# 02 — Preprocessing & Feature Engineering
## Support Ticket Priority Dataset (50K)

Ushbu notebookda: missing value ishlash, yangi featurelar yaratish, data leakage tekshiruvi, feature selection va train/test split bajariladi. Natijada tayyor `X_train/X_test/y_train/y_test` fayllari `../data/processed/` papkasiga saqlanadi va keyingi notebooklarda ishlatiladi.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

DATA_PATH = "../data/Support_tickets.csv"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.shape

(50000, 33)

## Missing values — to'ldirish

In [2]:
df["customer_sentiment"] = df["customer_sentiment"].fillna("Unknown")
df["customer_sentiment"].value_counts(dropna=False)

customer_sentiment
neutral     32944
negative    11317
positive     4833
Unknown       906
Name: count, dtype: int64

`customer_sentiment` dagi 906 ta missing qiymat `"Unknown"` kategoriyasi bilan to'ldirildi (01_EDA da asoslangan strategiya bo'yicha).


## 11. Feature Engineering

Quyida 5 ta yangi feature yaratiladi (talab qilingan minimal 3 tadan ko'proq):


In [3]:
# 1) Severity score — jiddiylikni ifodalovchi flag'larning yig'indisi
df["severity_flag_score"] = (
    df["payment_impact_flag"] + df["security_incident_flag"] +
    df["data_loss_flag"] + (df["has_runbook"] == 0).astype(int)
)

# 2) Impact per user — muammoning tashkilot hajmiga nisbatan og'irligi
df["impact_per_user"] = df["customers_affected"] / df["org_users"].replace(0, 1)

# 3) Recent activity ratio — oxirgi 30 kunlik ticketlarga nisbatan incidentlar ulushi
df["incident_to_ticket_ratio"] = df["past_90d_incidents"] / (df["past_30d_tickets"] + 1)

# 4) Downtime per affected customer — har bir ta'sirlangan mijozga to'g'ri keladigan downtime
df["downtime_per_customer"] = df["downtime_min"] / (df["customers_affected"] + 1)

# 5) Weekend flag — hafta oxirida ochilgan ticketlar (kuzatuv/qo'llab-quvvatlash sekinroq bo'lishi mumkin)
df["is_weekend"] = df["day_of_week"].isin(["Sat", "Sun"]).astype(int)

# 6) Long description flag — matn uzunligi bo'yicha ustunlik
df["is_long_description"] = (df["description_length"] > df["description_length"].median()).astype(int)

df[["severity_flag_score", "impact_per_user", "incident_to_ticket_ratio",
    "downtime_per_customer", "is_weekend", "is_long_description"]].describe()

,severity_flag_score,impact_per_user,incident_to_ticket_ratio,downtime_per_customer,is_weekend,is_long_description
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,0.457480,0.098715,0.501681,0.918277,0.050600,0.496660
std,0.517916,0.117223,0.556423,3.058743,0.219182,0.499994
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.030000,0.166667,0.000000,0.000000,0.000000
50%,0.000000,0.060796,0.333333,0.000000,0.000000,0.000000
75%,1.000000,0.120805,0.666667,0.466667,0.000000,1.000000
max,3.000000,1.000000,7.000000,102.000000,1.000000,1.000000


**Yangi featurelar tavsifi:**

1. **Feature nomi:** `severity_flag_score`
   **Nima qiladi:** `payment_impact_flag`, `security_incident_flag`, `data_loss_flag` va runbook yo'qligini bitta yig'indi ballga birlashtiradi (0-4 oralig'ida).
   **Nima uchun yordam beradi:** Bir nechta jiddiy hodisa flag'i alohida-alohida zaif signal bersa-da, yig'indi holida ticketning umumiy og'irlik darajasini yaxshiroq ifodalaydi.

2. **Feature nomi:** `impact_per_user`
   **Nima qiladi:** `customers_affected`ni tashkilotdagi umumiy foydalanuvchilar soniga nisbatlaydi.
   **Nima uchun yordam beradi:** Xom `customers_affected` soni katta kompaniyalar uchun tabiiy ravishda katta bo'ladi; nisbat muammoning **haqiqiy og'irligini** (masalan, butun tashkilot ishlamay qoldimi yoki faqat ozgina foydalanuvchi) yaxshiroq ko'rsatadi.

3. **Feature nomi:** `incident_to_ticket_ratio`
   **Nima qiladi:** Oxirgi 90 kunlik incidentlar sonini oxirgi 30 kunlik ticketlar soniga nisbatlaydi.
   **Nima uchun yordam beradi:** Bu "shovqin"dan (ko'p, lekin arzimas ticketlar) "haqiqiy muammo zichligi"ni ajratishga yordam beradi — yuqori nisbat muammolarning tez-tez jiddiy bo'lishi ehtimolini bildirishi mumkin.

4. **Feature nomi:** `downtime_per_customer`
   **Nima qiladi:** Downtime vaqtini ta'sirlangan mijozlar soniga nisbatlaydi.
   **Nima uchun yordam beradi:** Downtime va customers_affected orasidagi nisbiy og'irlikni ochib beradi — masalan, oz sonli mijozga juda uzoq downtime individual ravishda ikkalasidan ham ko'proq narsani anglatishi mumkin.

5. **Feature nomi:** `is_weekend`
   **Nima qiladi:** Ticket hafta oxirida (Sat/Sun) ochilganmi yoki yo'qmi (binary).
   **Nima uchun yordam beradi:** Hafta oxirida support jamoasi kamroq bo'lishi mumkin, bu ticketning qanday baholanishiga (yoki eskalatsiya qilinishiga) ta'sir qilishi mumkin.

6. **Feature nomi:** `is_long_description`
   **Nima qiladi:** Ticket tavsifi mediandan uzunroqmi (binary).
   **Nima uchun yordam beradi:** Uzunroq tavsiflar ko'pincha murakkabroq yoki jiddiyroq muammolarni tasvirlaydi, bu esa priority bilan bog'liq bo'lishi mumkin.


## 12. Data Leakage

**Savol:** Ushbu feature ticket priority aniqlangandan keyin paydo bo'ladigan ma'lumotni o'z ichiga oladimi?

Datasetdagi barcha columnlar ticket ochilgan payt yoki undan oldingi tarixiy ma'lumotlarni tasvirlaydi (masalan `past_30d_tickets`, `past_90d_incidents` — tarixiy; `customers_affected`, `downtime_min`, `error_rate_pct`, flag'lar — incidentning o'zi haqidagi darhol ma'lum bo'ladigan ma'lumotlar). Ushbu datasetda `final_resolution`, `resolved_at`, `resolution_time` kabi **ticket yopilgandan keyin** ma'lum bo'ladigan hech qanday column mavjud emas — shuning uchun to'g'ridan-to'g'ri "kelajakdan kelgan ma'lumot" (temporal leakage) yo'q.

Lekin bitta **strukturaviy (label) leakage** manbasi bor: **`priority_cat`** — bu `priority` (target) ning shunchaki raqamli kodi (1=low, 2=medium, 3=high), ya'ni target bilan 1:1 mos keladi. Buni modelga feature sifatida bersak, model targetni "aldab" 100% aniqlik bilan bashorat qiladi — bu klassik label leakage. Shuning uchun **`priority_cat` majburiy ravishda DROP qilinadi**.


In [4]:
assert (df["priority_cat"].map({1: "low", 2: "medium", 3: "high"}) == df["priority"]).all()
print("Tasdiqlandi: priority_cat = priority ning to'g'ridan-to'g'ri kodi -> DROP qilinadi (label leakage).")

Tasdiqlandi: priority_cat = priority ning to'g'ridan-to'g'ri kodi -> DROP qilinadi (label leakage).


## 13. Feature Selection

Quyidagi guruhlar DROP qilinadi:

- **ID columnlar:** `ticket_id` (unique identifier, har bir qatorda boshqacha — modelga hech qanday umumlashtiriladigan ma'lumot bermaydi), `company_id` (identifikator; garchi u qandaydir ma'noda kompaniya-xosligi haqida ma'lumot bersa-da, faqat 25 ta noyob qiymat train/testda tasodifiy taqsimlanib ketishi va overfittingga olib kelishi mumkin — shuning uchun uni ham drop qilamiz, o'rniga uning ta'siri allaqachon `company_size`, `industry`, `customer_tier`, `org_users` kabi tavsiflovchi featurelar orqali ifodalangan).
- **Label leakage:** `priority_cat` (target bilan bir xil ma'lumot).
- **Redundant pre-encoded columnlar:** `day_of_week_num`, `company_size_cat`, `industry_cat`, `customer_tier_cat`, `region_cat`, `product_area_cat`, `booking_channel_cat`, `reported_by_role_cat`, `customer_sentiment_cat` — bularning barchasi o'zlarining categorical (string) versiyalari bilan bir xil ma'lumotni takrorlaydi (label-encoded nusxalar). Modelga ikkala versiyani ham berish keraksiz duplikatsiya va (daraxt modellarida) noto'g'ri "ordinal" munosabat taxminiga olib kelishi mumkin. Original categorical ustunlarni saqlaymiz va keyin ularni to'g'ri (One-Hot) usulda encode qilamiz.

Constant yoki almost-constant columnlar tekshiriladi:


In [5]:
near_constant = []
for col in df.columns:
    top_freq = df[col].value_counts(normalize=True, dropna=False).iloc[0]
    if top_freq > 0.99:
        near_constant.append((col, round(top_freq, 4)))
print("Constant / almost-constant columnlar (>99% bir xil qiymat):", near_constant)

Constant / almost-constant columnlar (>99% bir xil qiymat): [('security_incident_flag', np.float64(0.9972)), ('data_loss_flag', np.float64(0.9949))]


In [6]:
drop_cols = [
    "ticket_id", "company_id",           # ID columns
    "priority_cat",                       # label leakage
    "day_of_week_num", "company_size_cat", "industry_cat", "customer_tier_cat",
    "region_cat", "product_area_cat", "booking_channel_cat",
    "reported_by_role_cat", "customer_sentiment_cat",   # redundant pre-encoded duplicates
]

model_df = df.drop(columns=drop_cols)
print("Qolgan columnlar:", list(model_df.columns))
print("Shape:", model_df.shape)

Qolgan columnlar: ['day_of_week', 'company_size', 'industry', 'customer_tier', 'org_users', 'region', 'past_30d_tickets', 'past_90d_incidents', 'product_area', 'booking_channel', 'reported_by_role', 'customers_affected', 'error_rate_pct', 'downtime_min', 'payment_impact_flag', 'security_incident_flag', 'data_loss_flag', 'has_runbook', 'customer_sentiment', 'description_length', 'priority', 'severity_flag_score', 'impact_per_user', 'incident_to_ticket_ratio', 'downtime_per_customer', 'is_weekend', 'is_long_description']
Shape: (50000, 27)


Almost-constant column topilmadi — barcha qolgan featurelar yetarlicha variativlikka ega, shuning uchun ular saqlanadi.


## 14. Train/Test Split

In [7]:
feature_cols = [c for c in model_df.columns if c != "priority"]
categorical_features = ["day_of_week", "company_size", "industry", "customer_tier",
                         "region", "product_area", "booking_channel", "reported_by_role",
                         "customer_sentiment"]
numerical_features = [c for c in feature_cols if c not in categorical_features]

X = model_df[feature_cols].copy()
y = model_df["priority"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).round(3))

Train shape: (40000, 26) Test shape: (10000, 26)

Train class distribution:
priority
low       0.50
medium    0.35
high      0.15
Name: proportion, dtype: float64

Test class distribution:
priority
low       0.50
medium    0.35
high      0.15
Name: proportion, dtype: float64


Dataset **80% train / 20% test** ga stratified split yordamida bo'lindi — bu Train va Test to'plamlarida class taqsimoti (Low/Medium/High nisbati) bir xil saqlanishini kafolatlaydi, bu esa imbalanced multiclass classification uchun muhim.

**Muhim:** Test dataset shu nuqtadan boshlab faqat yakuniy baholash uchun ishlatiladi — model tanlash va hyperparameter tuning FAQAT train (yoki cross-validation) yordamida amalga oshiriladi.


In [8]:
# One-Hot Encoding kategorik featurelar uchun (train ustida fit qilinadi, testga faqat transform)
X_train_enc = pd.get_dummies(X_train, columns=categorical_features, drop_first=False)
X_test_enc = pd.get_dummies(X_test, columns=categorical_features, drop_first=False)

# Testda train'da bo'lmagan/yo'q ustunlarni moslashtirish
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print("Encoded train shape:", X_train_enc.shape)
print("Encoded test shape:", X_test_enc.shape)

Encoded train shape: (40000, 59)
Encoded test shape: (10000, 59)


In [9]:
X_train_enc.to_parquet(f"{PROCESSED_DIR}/X_train.parquet")
X_test_enc.to_parquet(f"{PROCESSED_DIR}/X_test.parquet")
y_train.to_frame("priority").to_parquet(f"{PROCESSED_DIR}/y_train.parquet")
y_test.to_frame("priority").to_parquet(f"{PROCESSED_DIR}/y_test.parquet")

# Raqamli (label-encoded target, model kutubxonalari uchun qulay) versiyasi
label_map = {"low": 0, "medium": 1, "high": 2}
pd.Series(y_train.map(label_map), name="priority").to_frame().to_parquet(f"{PROCESSED_DIR}/y_train_num.parquet")
pd.Series(y_test.map(label_map), name="priority").to_frame().to_parquet(f"{PROCESSED_DIR}/y_test_num.parquet")

import json
with open(f"{PROCESSED_DIR}/label_map.json", "w") as f:
    json.dump(label_map, f)

print("Saqlandi:", os.listdir(PROCESSED_DIR))

Saqlandi: ['label_map.json', 'X_test.parquet', 'X_train.parquet', 'y_test.parquet', 'y_test_num.parquet', 'y_train.parquet', 'y_train_num.parquet']


## Xulosa

- Missing value (`customer_sentiment`) `"Unknown"` bilan to'ldirildi.
- 6 ta yangi feature yaratildi.
- Label leakage (`priority_cat`) va 9 ta redundant pre-encoded column, shuningdek ID columnlar (`ticket_id`, `company_id`) DROP qilindi.
- Dataset 80/20 stratified train/test ga bo'lindi va One-Hot encoding qo'llanildi.
- Tayyor fayllar `../data/processed/` papkasiga saqlandi — keyingi notebooklar ([03_Classical_Models.ipynb](03_Classical_Models.ipynb), [04_Boosting_Tuning.ipynb](04_Boosting_Tuning.ipynb)) shulardan foydalanadi.
